# 11 - Policy targeting under budget

## Causal question

If treatment is budget-constrained, what is the expected net gain from ranking individuals by predicted uplift versus random treatment assignment?

## Estimand

Expected net benefit under a fixed budget policy.

## Unit of analysis

Each row is one candidate for treatment.

## Identification assumptions

- No interference between units.
- The CATE model captures relevant effect heterogeneity and is transportable to the deployed population.
- Treatment assignment effects are stable in the evaluation window.

This notebook compares targeting policies when treatment is scarce.


## Decision framing

For fixed budget constraints, we compare three assignment rules:
- Random targeting
- Model-based targeting from predicted CATE
- Oracle targeting from true CATE (benchmark upper bound)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from causal_inference_lab.data_generators import make_heterogeneous_treatment_data
from causal_inference_lab.meta_learners import XMetaLearner

dataset = make_heterogeneous_treatment_data(n=5_000, seed=22)
data = dataset.data
covariates = ["age", "risk_score", "prior_usage"]


In [ ]:
budget = int(0.2 * len(data))
x = data[covariates]

# Oracle and model-based scores
oracle_score = data["true_ite"].to_numpy()
model = XMetaLearner().fit(
    data=data,
    covariates=covariates,
    treatment_col="treatment",
    outcome_col="outcome",
)
model_score = model.predict_cate(x)


In [ ]:
def expected_net_benefit(data: pd.DataFrame, selected: np.ndarray, cost_per_treat: float = 1.0) -> float:
    y = data["outcome"].to_numpy()
    tau = data["true_ite"].to_numpy()
    baseline = y.mean()
    treated_outcome = (y + tau)[selected == 1]
    untreated_outcome = y[selected == 0]
    return float(treated_outcome.mean() - cost_per_treat + untreated_outcome.mean())

rng = np.random.default_rng(11)
random_assignment = np.zeros(len(data), dtype=int)
random_assignment[rng.choice(len(data), size=budget, replace=False)] = 1

oracle_assignment = np.zeros(len(data), dtype=int)
oracle_assignment[np.argsort(oracle_score)[-budget:]] = 1

model_assignment = np.zeros(len(data), dtype=int)
model_assignment[np.argsort(model_score)[-budget:]] = 1

print(f"Random targeting net value: {expected_net_benefit(data, random_assignment):.3f}")
print(f"Model-based net value:  {expected_net_benefit(data, model_assignment):.3f}")
print(f"Oracle net value:       {expected_net_benefit(data, oracle_assignment):.3f}")


## Why accuracy matters

Ranking quality is usually more important than mean prediction quality. The same model can produce good average fit but poor lift in top budgeted units.

## Ethical and operational limitation

Policy targeting can amplify bias if the CATE model inherits historical inequities in covariates or treatment assignment. Always inspect fairness, stability, and counterfactual plausibility before deployment.
